# Notebook 01: Disagreement Profiling

**Question:** On which specific inputs do the full and optimised models disagree?

**Method:** Run all three model variants on the test set. For every input where variants disagree, record who was right.

**Variants compared:**
- `full` vs `onnx_fp32`
- `full` vs `onnx_int8`

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from quantscope import load_predictions, compute_disagreements, print_disagreement_report

In [ ]:
import os
os.makedirs('../results/charts', exist_ok=True)
print('✅ charts/ folder created')

In [ ]:
# Load predictions
data = load_predictions('../results/all_predictions.json')

texts  = data['texts']
labels = data['labels']

print(f'Test set   : {len(texts)} samples')
print(f'Label dist : {pd.Series(labels).value_counts().to_dict()}')

## Full vs ONNX fp32

In [ ]:
result_fp32 = compute_disagreements(data, 'full', 'onnx_fp32')
print_disagreement_report(result_fp32)

## Full vs ONNX INT8

In [ ]:
result_int8 = compute_disagreements(data, 'full', 'onnx_int8')
print_disagreement_report(result_int8)

## Inspect disagreement samples

In [ ]:
# Look at inputs where full was right but INT8 was wrong
hurt_indices = result_int8['a_correct_b_wrong'][:10]

rows = []
for idx in hurt_indices:
    rows.append({
        'text'         : texts[idx][:100],
        'true_label'   : labels[idx],
        'full_pred'    : data['full']['predictions'][idx],
        'full_conf'    : round(data['full']['confidences'][idx], 3),
        'int8_pred'    : data['onnx_int8']['predictions'][idx],
        'int8_conf'    : round(data['onnx_int8']['confidences'][idx], 3),
    })

pd.set_option('display.max_colwidth', 100)
pd.DataFrame(rows)

In [ ]:
# Look at inputs where INT8 was right but full was wrong
helped_indices = result_int8['b_correct_a_wrong'][:10]

rows = []
for idx in helped_indices:
    rows.append({
        'text'         : texts[idx][:100],
        'true_label'   : labels[idx],
        'full_pred'    : data['full']['predictions'][idx],
        'full_conf'    : round(data['full']['confidences'][idx], 3),
        'int8_pred'    : data['onnx_int8']['predictions'][idx],
        'int8_conf'    : round(data['onnx_int8']['confidences'][idx], 3),
    })

pd.DataFrame(rows)

In [ ]:
# Save disagreement indices for use in later notebooks
import json

summary = {
    'full_vs_fp32': {
        'disagreement_indices': result_fp32['a_correct_b_wrong'] + result_fp32['b_correct_a_wrong'] + result_fp32['both_wrong'],
        'hurt_indices'        : result_fp32['a_correct_b_wrong'],
        'helped_indices'      : result_fp32['b_correct_a_wrong'],
    },
    'full_vs_int8': {
        'disagreement_indices': result_int8['a_correct_b_wrong'] + result_int8['b_correct_a_wrong'] + result_int8['both_wrong'],
        'hurt_indices'        : result_int8['a_correct_b_wrong'],
        'helped_indices'      : result_int8['b_correct_a_wrong'],
    },
}

with open('../results/disagreement_summary.json', 'w') as f:
    json.dump(summary, f)

print('✅ Saved disagreement_summary.json')
print(f"  full vs fp32 disagreements : {len(summary['full_vs_fp32']['disagreement_indices'])}")
print(f"  full vs int8 disagreements : {len(summary['full_vs_int8']['disagreement_indices'])}")